# 06 - Dashboard Data Preparation

This notebook prepares all data needed for the Streamlit dashboard by computing KPIs, aggregating metrics, and creating export functions.

## Objectives
- Load data from PostgreSQL
- Calculate key performance indicators (KPIs)
- Prepare cleaned DataFrames for dashboard consumption
- Create export functions for dashboard modules
- Setup caching layer for performance

## Data Source
- PostgreSQL database: `walmart_fraud`
- Schema: `walmart_fraud`
- Period: January 2023 - December 2023
- Region: Central Florida

In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import json
import warnings

import sys
sys.path.insert(0, '..')

from src.database.connection import (
    load_orders, load_drivers, load_customers, load_products,
    load_missing_items, load_all_data, load_driver_metrics,
    load_customer_metrics, get_summary_stats, test_connection,
    execute_query
)
from src.features.aggregations import (
    get_overall_statistics, create_regional_features,
    create_driver_customer_matrix
)

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully!')

## 1. Data Loading

Load all data from PostgreSQL database.

In [ ]:
# Test database connection
if test_connection():
    print('Database connection: OK')
else:
    raise Exception('Database connection failed! Please check PostgreSQL is running.')

In [ ]:
# Load all data
print('Loading data from PostgreSQL...')

orders = load_orders()
print(f'  Orders: {len(orders):,} records')

drivers = load_drivers()
print(f'  Drivers: {len(drivers):,} records')

customers = load_customers()
print(f'  Customers: {len(customers):,} records')

products = load_products()
print(f'  Products: {len(products):,} records')

missing_items = load_missing_items()
print(f'  Missing Items: {len(missing_items):,} records')

print('\nData loading complete!')

In [ ]:
# Add derived features to orders
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['total_items'] = orders['items_delivered'] + orders['items_missing']
orders['missing_rate'] = np.where(
    orders['total_items'] > 0,
    (orders['items_missing'] / orders['total_items']) * 100,
    0
)
orders['has_missing'] = orders['items_missing'] > 0

# Temporal features
orders['month'] = orders['order_date'].dt.month
orders['month_name'] = orders['order_date'].dt.month_name()
orders['day_of_week'] = orders['order_date'].dt.day_name()
orders['day_of_week_num'] = orders['order_date'].dt.dayofweek

print('Derived features added to orders DataFrame')
orders.head()

## 2. KPI Calculations

Calculate all key performance indicators needed for the dashboard.

In [ ]:
def calculate_overview_metrics(orders_df: pd.DataFrame, drivers_df: pd.DataFrame, 
                               customers_df: pd.DataFrame) -> Dict:
    """
    Calculate overview metrics for dashboard.
    
    Args:
        orders_df: Orders DataFrame
        drivers_df: Drivers DataFrame
        customers_df: Customers DataFrame
        
    Returns:
        Dictionary with all key metrics
    """
    total_items = orders_df['items_delivered'].sum() + orders_df['items_missing'].sum()
    total_missing = orders_df['items_missing'].sum()
    orders_with_missing = (orders_df['items_missing'] > 0).sum()
    
    metrics = {
        # Order metrics
        'total_orders': int(len(orders_df)),
        'total_revenue': float(orders_df['order_amount'].sum()),
        'avg_order_value': float(orders_df['order_amount'].mean()),
        'median_order_value': float(orders_df['order_amount'].median()),
        'min_order_value': float(orders_df['order_amount'].min()),
        'max_order_value': float(orders_df['order_amount'].max()),
        
        # Item metrics
        'total_items_ordered': int(total_items),
        'total_items_delivered': int(orders_df['items_delivered'].sum()),
        'total_items_missing': int(total_missing),
        'overall_missing_rate': float((total_missing / total_items * 100) if total_items > 0 else 0),
        
        # Orders with issues
        'orders_with_missing': int(orders_with_missing),
        'pct_orders_with_missing': float((orders_with_missing / len(orders_df) * 100) if len(orders_df) > 0 else 0),
        
        # Entity counts
        'total_drivers': int(len(drivers_df)),
        'active_drivers': int(orders_df['driver_id'].nunique()),
        'total_customers': int(len(customers_df)),
        'active_customers': int(orders_df['customer_id'].nunique()),
        'total_regions': int(orders_df['region'].nunique()),
        
        # Time range
        'date_range_start': str(orders_df['order_date'].min().date()),
        'date_range_end': str(orders_df['order_date'].max().date()),
        
        # Estimated loss (assuming average $15 per missing item)
        'estimated_loss': float(total_missing * 15),
        
        # Timestamp
        'calculated_at': datetime.now().isoformat()
    }
    
    return metrics


# Calculate overview metrics
overview_metrics = calculate_overview_metrics(orders, drivers, customers)

print('OVERVIEW METRICS')
print('=' * 60)
for key, value in overview_metrics.items():
    if isinstance(value, float):
        print(f'{key}: {value:,.2f}')
    else:
        print(f'{key}: {value}')

In [ ]:
def calculate_driver_summary(orders_df: pd.DataFrame, drivers_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate driver performance summary.
    
    Args:
        orders_df: Orders DataFrame
        drivers_df: Drivers DataFrame
        
    Returns:
        DataFrame with driver summary metrics
    """
    # Aggregate orders by driver
    driver_orders = orders_df.groupby('driver_id').agg({
        'order_id': 'count',
        'order_amount': 'sum',
        'items_delivered': 'sum',
        'items_missing': 'sum',
        'has_missing': 'sum'
    }).reset_index()
    
    driver_orders.columns = ['driver_id', 'orders_completed', 'total_revenue', 
                             'items_delivered', 'items_missing', 'orders_with_missing']
    
    # Calculate metrics
    driver_orders['total_items'] = driver_orders['items_delivered'] + driver_orders['items_missing']
    driver_orders['missing_rate'] = np.where(
        driver_orders['total_items'] > 0,
        (driver_orders['items_missing'] / driver_orders['total_items']) * 100,
        0
    )
    driver_orders['pct_orders_with_missing'] = (
        driver_orders['orders_with_missing'] / driver_orders['orders_completed'] * 100
    )
    driver_orders['avg_order_value'] = driver_orders['total_revenue'] / driver_orders['orders_completed']
    
    # Merge with driver info
    driver_summary = drivers_df.merge(driver_orders, on='driver_id', how='left')
    
    # Fill NaN for drivers with no orders
    driver_summary = driver_summary.fillna(0)
    
    # Calculate risk score (0-100 scale)
    # Risk factors: high missing rate, high % orders with missing
    max_missing_rate = driver_summary['missing_rate'].max()
    max_pct_orders = driver_summary['pct_orders_with_missing'].max()
    
    driver_summary['risk_score'] = np.where(
        (max_missing_rate > 0) & (max_pct_orders > 0),
        (
            (driver_summary['missing_rate'] / max_missing_rate * 50) +
            (driver_summary['pct_orders_with_missing'] / max_pct_orders * 50)
        ),
        0
    )
    
    # Risk category
    driver_summary['risk_category'] = pd.cut(
        driver_summary['risk_score'],
        bins=[-1, 25, 50, 75, 100],
        labels=['Low', 'Medium', 'High', 'Critical']
    )
    
    return driver_summary.sort_values('risk_score', ascending=False)


# Calculate driver summary
driver_summary = calculate_driver_summary(orders, drivers)

print(f'Driver Summary: {len(driver_summary)} drivers')
print('\nTop 10 Highest Risk Drivers:')
driver_summary.head(10)

In [ ]:
def calculate_customer_summary(orders_df: pd.DataFrame, customers_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate customer behavior summary.
    
    Args:
        orders_df: Orders DataFrame
        customers_df: Customers DataFrame
        
    Returns:
        DataFrame with customer summary metrics
    """
    # Aggregate orders by customer
    customer_orders = orders_df.groupby('customer_id').agg({
        'order_id': 'count',
        'order_amount': 'sum',
        'items_delivered': 'sum',
        'items_missing': 'sum',
        'has_missing': 'sum'
    }).reset_index()
    
    customer_orders.columns = ['customer_id', 'total_orders', 'total_spent', 
                               'items_received', 'items_reported_missing', 'orders_with_claims']
    
    # Calculate metrics
    customer_orders['total_items'] = customer_orders['items_received'] + customer_orders['items_reported_missing']
    customer_orders['claim_rate'] = np.where(
        customer_orders['total_items'] > 0,
        (customer_orders['items_reported_missing'] / customer_orders['total_items']) * 100,
        0
    )
    customer_orders['pct_orders_with_claims'] = (
        customer_orders['orders_with_claims'] / customer_orders['total_orders'] * 100
    )
    customer_orders['avg_order_value'] = customer_orders['total_spent'] / customer_orders['total_orders']
    
    # Merge with customer info
    customer_summary = customers_df.merge(customer_orders, on='customer_id', how='left')
    
    # Fill NaN for customers with no orders
    customer_summary = customer_summary.fillna(0)
    
    # Calculate risk score (0-100 scale)
    max_claim_rate = customer_summary['claim_rate'].max()
    max_pct_orders = customer_summary['pct_orders_with_claims'].max()
    
    customer_summary['risk_score'] = np.where(
        (max_claim_rate > 0) & (max_pct_orders > 0),
        (
            (customer_summary['claim_rate'] / max_claim_rate * 50) +
            (customer_summary['pct_orders_with_claims'] / max_pct_orders * 50)
        ),
        0
    )
    
    # Risk category
    customer_summary['risk_category'] = pd.cut(
        customer_summary['risk_score'],
        bins=[-1, 25, 50, 75, 100],
        labels=['Low', 'Medium', 'High', 'Critical']
    )
    
    # Customer segment based on spending
    customer_summary['spending_segment'] = pd.cut(
        customer_summary['total_spent'],
        bins=[-1, 500, 2000, 5000, float('inf')],
        labels=['Low Value', 'Medium Value', 'High Value', 'Premium']
    )
    
    return customer_summary.sort_values('risk_score', ascending=False)


# Calculate customer summary
customer_summary = calculate_customer_summary(orders, customers)

print(f'Customer Summary: {len(customer_summary)} customers')
print('\nTop 10 Highest Risk Customers:')
customer_summary.head(10)

In [ ]:
def calculate_regional_summary(orders_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate regional performance summary.
    
    Args:
        orders_df: Orders DataFrame
        
    Returns:
        DataFrame with regional metrics
    """
    # Aggregate by region
    regional = orders_df.groupby('region').agg({
        'order_id': 'count',
        'order_amount': ['sum', 'mean'],
        'items_delivered': 'sum',
        'items_missing': 'sum',
        'driver_id': 'nunique',
        'customer_id': 'nunique',
        'has_missing': 'sum'
    }).reset_index()
    
    regional.columns = ['region', 'total_orders', 'total_revenue', 'avg_order_value',
                        'items_delivered', 'items_missing', 'unique_drivers', 
                        'unique_customers', 'orders_with_missing']
    
    # Calculate metrics
    regional['total_items'] = regional['items_delivered'] + regional['items_missing']
    regional['missing_rate'] = np.where(
        regional['total_items'] > 0,
        (regional['items_missing'] / regional['total_items']) * 100,
        0
    )
    regional['pct_orders_with_missing'] = (
        regional['orders_with_missing'] / regional['total_orders'] * 100
    )
    regional['orders_per_driver'] = regional['total_orders'] / regional['unique_drivers']
    regional['orders_per_customer'] = regional['total_orders'] / regional['unique_customers']
    
    # Revenue share
    regional['revenue_share'] = regional['total_revenue'] / regional['total_revenue'].sum() * 100
    
    # Risk ranking
    regional['risk_rank'] = regional['missing_rate'].rank(ascending=False).astype(int)
    
    return regional.sort_values('missing_rate', ascending=False)


# Calculate regional summary
regional_summary = calculate_regional_summary(orders)

print(f'Regional Summary: {len(regional_summary)} regions')
regional_summary

In [ ]:
def calculate_temporal_trends(orders_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Calculate temporal trends for dashboard.
    
    Args:
        orders_df: Orders DataFrame
        
    Returns:
        Dictionary with temporal DataFrames
    """
    # Monthly trends
    monthly = orders_df.groupby('month').agg({
        'order_id': 'count',
        'order_amount': 'sum',
        'items_delivered': 'sum',
        'items_missing': 'sum',
        'has_missing': 'sum'
    }).reset_index()
    
    monthly.columns = ['month', 'orders', 'revenue', 'items_delivered', 'items_missing', 'orders_with_missing']
    monthly['missing_rate'] = (
        monthly['items_missing'] / (monthly['items_delivered'] + monthly['items_missing']) * 100
    )
    monthly['pct_orders_with_missing'] = monthly['orders_with_missing'] / monthly['orders'] * 100
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    monthly['month_name'] = monthly['month'].apply(lambda x: month_names[x-1])
    
    # Daily trends (day of week)
    daily = orders_df.groupby(['day_of_week', 'day_of_week_num']).agg({
        'order_id': 'count',
        'order_amount': 'sum',
        'items_delivered': 'sum',
        'items_missing': 'sum'
    }).reset_index().sort_values('day_of_week_num')
    
    daily.columns = ['day_of_week', 'day_num', 'orders', 'revenue', 'items_delivered', 'items_missing']
    daily['missing_rate'] = (
        daily['items_missing'] / (daily['items_delivered'] + daily['items_missing']) * 100
    )
    
    # Hourly trends
    hourly = orders_df.groupby('delivery_hour').agg({
        'order_id': 'count',
        'items_delivered': 'sum',
        'items_missing': 'sum'
    }).reset_index()
    
    hourly.columns = ['hour', 'orders', 'items_delivered', 'items_missing']
    hourly['missing_rate'] = (
        hourly['items_missing'] / (hourly['items_delivered'] + hourly['items_missing']) * 100
    )
    
    # Define periods
    def get_period(hour):
        if 6 <= hour < 12:
            return 'Morning'
        elif 12 <= hour < 18:
            return 'Afternoon'
        elif 18 <= hour < 22:
            return 'Evening'
        else:
            return 'Night'
    
    hourly['period'] = hourly['hour'].apply(get_period)
    
    return {
        'monthly': monthly,
        'daily': daily,
        'hourly': hourly
    }


# Calculate temporal trends
temporal_trends = calculate_temporal_trends(orders)

print('MONTHLY TRENDS')
display(temporal_trends['monthly'])

print('\nDAILY TRENDS (Day of Week)')
display(temporal_trends['daily'])

print('\nHOURLY TRENDS')
display(temporal_trends['hourly'].head(10))

In [ ]:
def calculate_risk_distribution(driver_summary: pd.DataFrame, 
                                customer_summary: pd.DataFrame) -> Dict:
    """
    Calculate risk distribution for drivers and customers.
    
    Args:
        driver_summary: Driver summary DataFrame
        customer_summary: Customer summary DataFrame
        
    Returns:
        Dictionary with risk distribution data
    """
    # Driver risk distribution
    driver_risk_dist = driver_summary['risk_category'].value_counts().to_dict()
    
    # Customer risk distribution
    customer_risk_dist = customer_summary['risk_category'].value_counts().to_dict()
    
    return {
        'driver_risk_distribution': {
            'Low': int(driver_risk_dist.get('Low', 0)),
            'Medium': int(driver_risk_dist.get('Medium', 0)),
            'High': int(driver_risk_dist.get('High', 0)),
            'Critical': int(driver_risk_dist.get('Critical', 0))
        },
        'customer_risk_distribution': {
            'Low': int(customer_risk_dist.get('Low', 0)),
            'Medium': int(customer_risk_dist.get('Medium', 0)),
            'High': int(customer_risk_dist.get('High', 0)),
            'Critical': int(customer_risk_dist.get('Critical', 0))
        }
    }


# Calculate risk distribution
risk_distribution = calculate_risk_distribution(driver_summary, customer_summary)

print('RISK DISTRIBUTION')
print('=' * 40)
print('\nDriver Risk Distribution:')
for category, count in risk_distribution['driver_risk_distribution'].items():
    print(f'  {category}: {count}')
    
print('\nCustomer Risk Distribution:')
for category, count in risk_distribution['customer_risk_distribution'].items():
    print(f'  {category}: {count}')

## 3. Dashboard Datasets

Create cleaned DataFrames ready for dashboard consumption.

In [ ]:
def generate_risk_alerts(driver_summary: pd.DataFrame, 
                         customer_summary: pd.DataFrame,
                         regional_summary: pd.DataFrame,
                         threshold_score: float = 70.0) -> pd.DataFrame:
    """
    Generate risk alerts for high-risk entities.
    
    Args:
        driver_summary: Driver summary DataFrame
        customer_summary: Customer summary DataFrame
        regional_summary: Regional summary DataFrame
        threshold_score: Minimum risk score to generate alert
        
    Returns:
        DataFrame with risk alerts
    """
    alerts = []
    
    # High-risk drivers
    high_risk_drivers = driver_summary[driver_summary['risk_score'] >= threshold_score]
    for _, row in high_risk_drivers.iterrows():
        alerts.append({
            'entity_type': 'Driver',
            'entity_id': row['driver_id'],
            'entity_name': row['driver_name'],
            'risk_score': row['risk_score'],
            'risk_category': row['risk_category'],
            'primary_metric': f"Missing rate: {row['missing_rate']:.2f}%",
            'secondary_metric': f"Orders with issues: {row['orders_with_missing']:.0f}",
            'recommendation': 'Review delivery patterns and consider audit'
        })
    
    # High-risk customers
    high_risk_customers = customer_summary[customer_summary['risk_score'] >= threshold_score]
    for _, row in high_risk_customers.iterrows():
        alerts.append({
            'entity_type': 'Customer',
            'entity_id': row['customer_id'],
            'entity_name': row['customer_name'],
            'risk_score': row['risk_score'],
            'risk_category': row['risk_category'],
            'primary_metric': f"Claim rate: {row['claim_rate']:.2f}%",
            'secondary_metric': f"Orders with claims: {row['orders_with_claims']:.0f}",
            'recommendation': 'Verify claims and consider enhanced verification'
        })
    
    # High-risk regions (top 3 by missing rate)
    overall_missing_rate = regional_summary['missing_rate'].mean()
    high_risk_regions = regional_summary[regional_summary['missing_rate'] > overall_missing_rate * 1.2]
    for _, row in high_risk_regions.iterrows():
        alerts.append({
            'entity_type': 'Region',
            'entity_id': row['region'],
            'entity_name': row['region'],
            'risk_score': row['missing_rate'],  # Using missing rate as score for regions
            'risk_category': 'High' if row['missing_rate'] > overall_missing_rate * 1.5 else 'Medium',
            'primary_metric': f"Missing rate: {row['missing_rate']:.2f}%",
            'secondary_metric': f"Items missing: {row['items_missing']:.0f}",
            'recommendation': 'Investigate regional patterns and driver assignments'
        })
    
    alerts_df = pd.DataFrame(alerts)
    if len(alerts_df) > 0:
        alerts_df = alerts_df.sort_values('risk_score', ascending=False)
    
    return alerts_df


# Generate risk alerts
risk_alerts = generate_risk_alerts(driver_summary, customer_summary, regional_summary)

print(f'Total Risk Alerts: {len(risk_alerts)}')
print('\nAlert Summary by Entity Type:')
if len(risk_alerts) > 0:
    print(risk_alerts['entity_type'].value_counts())
    print('\nTop 10 Alerts:')
    display(risk_alerts.head(10))
else:
    print('No high-risk alerts generated (threshold: 70)')

In [ ]:
def get_top_suspicious_entities(driver_summary: pd.DataFrame,
                                customer_summary: pd.DataFrame,
                                n: int = 10) -> Dict[str, pd.DataFrame]:
    """
    Get top suspicious drivers and customers.
    
    Args:
        driver_summary: Driver summary DataFrame
        customer_summary: Customer summary DataFrame
        n: Number of top entities to return
        
    Returns:
        Dictionary with top suspicious entities
    """
    top_drivers = driver_summary.nlargest(n, 'risk_score')[[
        'driver_id', 'driver_name', 'age', 'orders_completed',
        'missing_rate', 'pct_orders_with_missing', 'risk_score', 'risk_category'
    ]]
    
    top_customers = customer_summary.nlargest(n, 'risk_score')[[
        'customer_id', 'customer_name', 'customer_age', 'total_orders',
        'claim_rate', 'pct_orders_with_claims', 'risk_score', 'risk_category'
    ]]
    
    return {
        'top_suspicious_drivers': top_drivers,
        'top_suspicious_customers': top_customers
    }


# Get top suspicious entities
top_suspicious = get_top_suspicious_entities(driver_summary, customer_summary)

print('TOP 10 SUSPICIOUS DRIVERS')
display(top_suspicious['top_suspicious_drivers'])

print('\nTOP 10 SUSPICIOUS CUSTOMERS')
display(top_suspicious['top_suspicious_customers'])

In [ ]:
def get_product_analysis(missing_items_df: pd.DataFrame, products_df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze products that are most frequently reported as missing.
    
    Args:
        missing_items_df: Missing items DataFrame
        products_df: Products DataFrame
        
    Returns:
        DataFrame with product analysis
    """
    # Count missing reports per product
    product_missing = missing_items_df.groupby('product_id').agg({
        'missing_item_id': 'count',
        'order_id': 'nunique'
    }).reset_index()
    
    product_missing.columns = ['product_id', 'times_reported_missing', 'orders_affected']
    
    # Merge with product info
    product_analysis = products_df.merge(product_missing, on='product_id', how='left')
    product_analysis['times_reported_missing'] = product_analysis['times_reported_missing'].fillna(0).astype(int)
    product_analysis['orders_affected'] = product_analysis['orders_affected'].fillna(0).astype(int)
    
    # Calculate total estimated loss per product
    product_analysis['estimated_loss'] = product_analysis['times_reported_missing'] * product_analysis['price']
    
    return product_analysis.sort_values('times_reported_missing', ascending=False)


# Analyze products
product_analysis = get_product_analysis(missing_items, products)

print('TOP 20 MOST FREQUENTLY MISSING PRODUCTS')
display(product_analysis.head(20))

## 4. Export Functions

Create functions that the dashboard can import to get prepared data.

In [ ]:
# Define all export functions that will be used by the dashboard

def get_overview_metrics() -> Dict:
    """
    Get overview metrics for dashboard main page.
    
    Returns:
        Dictionary with all key metrics
    """
    orders = load_orders()
    drivers = load_drivers()
    customers = load_customers()
    
    # Add derived columns
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    orders['has_missing'] = orders['items_missing'] > 0
    
    return calculate_overview_metrics(orders, drivers, customers)


def get_driver_summary() -> pd.DataFrame:
    """
    Get driver performance summary.
    
    Returns:
        DataFrame with driver summary
    """
    orders = load_orders()
    drivers = load_drivers()
    
    orders['has_missing'] = orders['items_missing'] > 0
    
    return calculate_driver_summary(orders, drivers)


def get_customer_summary() -> pd.DataFrame:
    """
    Get customer behavior summary.
    
    Returns:
        DataFrame with customer summary
    """
    orders = load_orders()
    customers = load_customers()
    
    orders['has_missing'] = orders['items_missing'] > 0
    
    return calculate_customer_summary(orders, customers)


def get_regional_summary() -> pd.DataFrame:
    """
    Get regional performance summary.
    
    Returns:
        DataFrame with regional metrics
    """
    orders = load_orders()
    orders['has_missing'] = orders['items_missing'] > 0
    
    return calculate_regional_summary(orders)


def get_temporal_trends() -> Dict[str, pd.DataFrame]:
    """
    Get temporal trends data.
    
    Returns:
        Dictionary with monthly, daily, and hourly DataFrames
    """
    orders = load_orders()
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    orders['month'] = orders['order_date'].dt.month
    orders['day_of_week'] = orders['order_date'].dt.day_name()
    orders['day_of_week_num'] = orders['order_date'].dt.dayofweek
    orders['has_missing'] = orders['items_missing'] > 0
    
    return calculate_temporal_trends(orders)


def get_risk_alerts(threshold: float = 70.0) -> pd.DataFrame:
    """
    Get high-risk alerts for dashboard.
    
    Args:
        threshold: Minimum risk score for alerts
        
    Returns:
        DataFrame with risk alerts
    """
    driver_summary = get_driver_summary()
    customer_summary = get_customer_summary()
    regional_summary = get_regional_summary()
    
    return generate_risk_alerts(driver_summary, customer_summary, regional_summary, threshold)


def get_risk_distribution() -> Dict:
    """
    Get risk distribution for drivers and customers.
    
    Returns:
        Dictionary with risk distribution data
    """
    driver_summary = get_driver_summary()
    customer_summary = get_customer_summary()
    
    return calculate_risk_distribution(driver_summary, customer_summary)


def get_top_suspicious(n: int = 10) -> Dict[str, pd.DataFrame]:
    """
    Get top suspicious entities.
    
    Args:
        n: Number of entities to return
        
    Returns:
        Dictionary with top suspicious drivers and customers
    """
    driver_summary = get_driver_summary()
    customer_summary = get_customer_summary()
    
    return get_top_suspicious_entities(driver_summary, customer_summary, n)


def get_product_summary() -> pd.DataFrame:
    """
    Get product analysis summary.
    
    Returns:
        DataFrame with product analysis
    """
    missing_items = load_missing_items()
    products = load_products()
    
    return get_product_analysis(missing_items, products)


print('Export functions defined successfully!')
print('\nAvailable functions:')
print('  - get_overview_metrics() -> Dict')
print('  - get_driver_summary() -> pd.DataFrame')
print('  - get_customer_summary() -> pd.DataFrame')
print('  - get_regional_summary() -> pd.DataFrame')
print('  - get_temporal_trends() -> Dict[str, pd.DataFrame]')
print('  - get_risk_alerts(threshold: float) -> pd.DataFrame')
print('  - get_risk_distribution() -> Dict')
print('  - get_top_suspicious(n: int) -> Dict[str, pd.DataFrame]')
print('  - get_product_summary() -> pd.DataFrame')

## 5. Cache Layer

The cache layer is implemented in `src/dashboard/data_cache.py`. This section demonstrates how to use it.

In [ ]:
# Demonstrate the cache layer structure
# The actual implementation is in src/dashboard/data_cache.py

cache_module_code = '''
"""
Dashboard Data Cache Module

Provides caching layer for dashboard data to improve performance.
Uses functools.lru_cache and time-based cache invalidation.
"""

Key features:
1. LRU Cache with configurable TTL (Time To Live)
2. Thread-safe caching for Streamlit multi-user scenarios
3. Lazy loading - data is only loaded when first requested
4. Manual cache refresh capabilities
5. Memory-efficient with automatic cache cleanup

Usage in dashboard:
    from src.dashboard.data_cache import DashboardCache
    
    cache = DashboardCache(ttl_minutes=15)
    
    # Get cached data
    metrics = cache.get_overview_metrics()
    drivers = cache.get_driver_summary()
    
    # Force refresh
    cache.refresh_all()
'''

print('Cache Layer Overview')
print('=' * 60)
print(cache_module_code)

In [ ]:
# Test that the cache module can be imported
try:
    from src.dashboard.data_cache import DashboardCache
    
    # Initialize cache with 15-minute TTL
    cache = DashboardCache(ttl_minutes=15)
    
    print('Cache module loaded successfully!')
    print(f'Cache TTL: {cache.ttl_minutes} minutes')
    
    # Test getting overview metrics
    print('\nTesting cache.get_overview_metrics()...')
    metrics = cache.get_overview_metrics()
    print(f'  Total Orders: {metrics["total_orders"]:,}')
    print(f'  Total Revenue: ${metrics["total_revenue"]:,.2f}')
    print(f'  Overall Missing Rate: {metrics["overall_missing_rate"]:.2f}%')
    
except ImportError as e:
    print(f'Note: Cache module not yet created. Error: {e}')
    print('Run this notebook after creating src/dashboard/data_cache.py')

## 6. Data Validation

Validate all prepared data before exporting.

In [ ]:
def validate_dashboard_data():
    """
    Validate all dashboard data is correctly prepared.
    """
    print('DATA VALIDATION')
    print('=' * 60)
    
    validation_results = []
    
    # Validate overview metrics
    print('\n1. Overview Metrics')
    required_metrics = [
        'total_orders', 'total_revenue', 'overall_missing_rate',
        'orders_with_missing', 'active_drivers', 'active_customers'
    ]
    missing_metrics = [m for m in required_metrics if m not in overview_metrics]
    if missing_metrics:
        print(f'   FAIL: Missing metrics: {missing_metrics}')
        validation_results.append(False)
    else:
        print(f'   PASS: All {len(required_metrics)} required metrics present')
        validation_results.append(True)
    
    # Validate driver summary
    print('\n2. Driver Summary')
    required_driver_cols = [
        'driver_id', 'driver_name', 'orders_completed', 
        'missing_rate', 'risk_score', 'risk_category'
    ]
    missing_cols = [c for c in required_driver_cols if c not in driver_summary.columns]
    if missing_cols:
        print(f'   FAIL: Missing columns: {missing_cols}')
        validation_results.append(False)
    else:
        print(f'   PASS: All {len(required_driver_cols)} required columns present')
        print(f'   Records: {len(driver_summary):,}')
        validation_results.append(True)
    
    # Validate customer summary
    print('\n3. Customer Summary')
    required_customer_cols = [
        'customer_id', 'customer_name', 'total_orders',
        'claim_rate', 'risk_score', 'risk_category'
    ]
    missing_cols = [c for c in required_customer_cols if c not in customer_summary.columns]
    if missing_cols:
        print(f'   FAIL: Missing columns: {missing_cols}')
        validation_results.append(False)
    else:
        print(f'   PASS: All {len(required_customer_cols)} required columns present')
        print(f'   Records: {len(customer_summary):,}')
        validation_results.append(True)
    
    # Validate regional summary
    print('\n4. Regional Summary')
    required_regional_cols = [
        'region', 'total_orders', 'total_revenue', 
        'missing_rate', 'unique_drivers'
    ]
    missing_cols = [c for c in required_regional_cols if c not in regional_summary.columns]
    if missing_cols:
        print(f'   FAIL: Missing columns: {missing_cols}')
        validation_results.append(False)
    else:
        print(f'   PASS: All {len(required_regional_cols)} required columns present')
        print(f'   Regions: {len(regional_summary)}')
        validation_results.append(True)
    
    # Validate temporal trends
    print('\n5. Temporal Trends')
    required_trend_keys = ['monthly', 'daily', 'hourly']
    missing_keys = [k for k in required_trend_keys if k not in temporal_trends]
    if missing_keys:
        print(f'   FAIL: Missing trend data: {missing_keys}')
        validation_results.append(False)
    else:
        print(f'   PASS: All temporal trend DataFrames present')
        for key, df in temporal_trends.items():
            print(f'     - {key}: {len(df)} records')
        validation_results.append(True)
    
    # Validate risk alerts
    print('\n6. Risk Alerts')
    if len(risk_alerts) > 0:
        required_alert_cols = [
            'entity_type', 'entity_id', 'risk_score', 'risk_category'
        ]
        missing_cols = [c for c in required_alert_cols if c not in risk_alerts.columns]
        if missing_cols:
            print(f'   FAIL: Missing columns: {missing_cols}')
            validation_results.append(False)
        else:
            print(f'   PASS: {len(risk_alerts)} alerts generated with correct structure')
            validation_results.append(True)
    else:
        print(f'   PASS: No alerts generated (may be normal if no high-risk entities)')
        validation_results.append(True)
    
    # Summary
    print('\n' + '=' * 60)
    if all(validation_results):
        print('VALIDATION PASSED: All dashboard data is correctly prepared!')
    else:
        print(f'VALIDATION FAILED: {validation_results.count(False)} issues found')
    
    return all(validation_results)


# Run validation
validation_passed = validate_dashboard_data()

## Summary

### Data Prepared for Dashboard:

1. **Overview Metrics** (`overview_metrics`)
   - Total orders, revenue, items
   - Missing rates and estimated losses
   - Entity counts (drivers, customers, regions)
   - Date range

2. **Driver Summary** (`driver_summary`)
   - Driver performance metrics
   - Risk scores and categories
   - Orders completed, missing rates

3. **Customer Summary** (`customer_summary`)
   - Customer behavior metrics
   - Claim rates and risk scores
   - Spending segments

4. **Regional Summary** (`regional_summary`)
   - Regional performance metrics
   - Missing rates by region
   - Driver/customer distribution

5. **Temporal Trends** (`temporal_trends`)
   - Monthly trends
   - Day of week patterns
   - Hourly patterns

6. **Risk Alerts** (`risk_alerts`)
   - High-risk drivers
   - High-risk customers
   - High-risk regions

### Export Functions:
- `get_overview_metrics()` - Dashboard main metrics
- `get_driver_summary()` - Driver analysis data
- `get_customer_summary()` - Customer analysis data
- `get_regional_summary()` - Regional analysis data
- `get_temporal_trends()` - Time series data
- `get_risk_alerts()` - High-risk entity alerts
- `get_risk_distribution()` - Risk category distribution
- `get_top_suspicious()` - Top suspicious entities
- `get_product_summary()` - Product analysis data

### Cache Layer:
- Implemented in `src/dashboard/data_cache.py`
- LRU cache with configurable TTL
- Thread-safe for multi-user dashboard scenarios